# Instalar dependencias

In [5]:
!pip install pandas
!pip install xlrd


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip

[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: pip install --upgrade pip


# Limpiando los datos

## Limpieza y unión Tabla 1 y Tabla 2

In [29]:
import pandas as pd

def clean_data(tabla_1, tabla_2):
    # Limpiar tabla_1
    tabla_1 = tabla_1[['Ass Codnum', 'Tipologia', 'Curso']]
    tabla_1 = tabla_1.rename(columns={'Ass Codnum': 'Codigo'})
    tabla_1 = tabla_1.drop_duplicates()

    # Limpiar tabla_2
    tabla_2 = tabla_2[['Curso Aca', 'Cod Asig', 'Nummat', 'Asignatura', 'Tasa Rend', 'Tasa Exito']]
    tabla_2 = tabla_2.rename(columns=
    {'Curso Aca':'Anio', 
    'Cod Asig': 'Codigo', 
    'Nummat': 'Matriculados',
    'Tasa Rend': 'Tasa_Rendimiento',
    'Tasa Exito': 'Tasa_Exito',
    })
    tabla_2 = tabla_2.drop_duplicates()
    print(tabla_2.columns)
    
    # Join Tabla_1 y Tabla_2
    df = pd.merge(tabla_1, tabla_2, on='Codigo', how='left')
    
    return df


tabla_1 = pd.read_excel('../data/raw_data/Tabla_01_Asignaturas_plan_pdi.xls', header=4)
tabla_2 = pd.read_excel('../data/raw_data/Tabla_02_Resultados_asignaturas_plan.xls', header=4)
# tabla_1.head()
# tabla_2.head()
df = clean_data(tabla_1, tabla_2)
print(df.head())

Index(['Anio', 'Codigo', 'Matriculados', 'Asignatura', 'Tasa_Rendimiento',
       'Tasa_Exito'],
      dtype='object')
      Codigo         Tipologia  Curso     Anio  Matriculados  \
0  139261011  FORMACIÓN BÁSICA      1  2012-13           130   
1  139261011  FORMACIÓN BÁSICA      1  2023-24           255   
2  139261011  FORMACIÓN BÁSICA      1  2020-21           145   
3  139261011  FORMACIÓN BÁSICA      1  2015-16           147   
4  139261011  FORMACIÓN BÁSICA      1  2017-18           178   

           Asignatura  Tasa_Rendimiento  Tasa_Exito  
0  INFORMÁTICA BÁSICA              68.5        84.8  
1  INFORMÁTICA BÁSICA              50.6        72.9  
2  INFORMÁTICA BÁSICA              33.1        47.1  
3  INFORMÁTICA BÁSICA              61.9        74.0  
4  INFORMÁTICA BÁSICA              75.8        86.0  


## Limpieza tabla 4

In [35]:
df = pd.read_excel('../data/raw_data/Tabla_04_Evolución_indicadores.xls', header=5)

# 1. Cargamos el archivo saltando las primeras 5 líneas de encabezado/basura

# 2. Renombramos la primera columna para que sea más fácil trabajar con ella
df = df.rename(columns={'Unnamed: 0': 'Indicador'})

# 3. Creamos un diccionario para mapear los nombres largos originales a los nombres limpios que quieres
mapeo_columnas = {
    "18   -Tasa de éxito del título": "Tasa éxito",
    "15   -Tasa de abandono del título - (IA)": "Tasa abandono",
    "17   -Tasa de rendimiento del título - (IA)": "Tasa rendimiento",
    "16   -Tasa de eficiencia de los graduados - (IA)": "Tasa eficiencia",
    "14   -Tasa de graduación del título - (IA)": "Tasa graduación"
}

# 4. Filtramos el DataFrame para quedarnos solo con esas filas
df_filtrado = df[df['Indicador'].isin(mapeo_columnas.keys())].copy()

# 5. Aplicamos los nombres limpios a la columna de indicadores
df_filtrado['Indicador'] = df_filtrado['Indicador'].map(mapeo_columnas)

# 6. Ponemos la columna 'Indicador' como el índice del DataFrame
df_filtrado.set_index('Indicador', inplace=True)

# 7. ¡LA MAGIA! Transponemos la tabla (giramos filas por columnas)
df_transpuesto = df_filtrado.transpose()

# 8. Reiniciamos el índice para que los años pasen a ser una columna normal llamada "Anio"
df_transpuesto.reset_index(inplace=True)
df_transpuesto.rename(columns={'index': 'Anio'}, inplace=True)
df_transpuesto.columns.name = None # Quitamos el nombre del eje de columnas por limpieza

# Guardamos el resultado en un nuevo archivo
print(df_transpuesto)

       Anio  Tasa graduación  Tasa abandono  Tasa eficiencia  \
0   2008-09              NaN            NaN              NaN   
1   2009-10              NaN            NaN              NaN   
2   2010-11             18.6           26.3              NaN   
3   2011-12             32.7           31.0              NaN   
4   2012-13             29.2           24.5              NaN   
5   2013-14             28.2           23.7             94.9   
6   2014-15             30.7           26.8             88.7   
7   2015-16             23.8           28.5             84.6   
8   2016-17             22.4           24.3             85.0   
9   2017-18             26.7           20.0             84.4   
10  2018-19             38.0           24.7             80.5   
11  2019-20             34.6           17.0             79.0   
12  2020-21             35.5           21.7             81.2   
13  2021-22              NaN           17.7             83.4   
14  2022-23              NaN           1